In [1]:
# CELDA 1: Instalación de librerías necesarias
!pip install requests pandas openpyxl tqdm google-search-results -q

print("✅ Librerías instaladas correctamente")

  Preparing metadata (setup.py) ... done
✅ Librerías instaladas correctamente


In [15]:
# CELDA 2: Configuración general
import time, os
from datetime import datetime

# --------------------------------------------------------------------
# 🔑 1. Tu clave de SerpApi (https://serpapi.com/manage-api-key)
SERPAPI_KEY = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <-- reemplaza esto

# --------------------------------------------------------------------
# ⚙️ 2. Límites de seguridad
MAX_BUSQUEDAS_SERPAPI = 10
TIMEOUT_REQUEST = 20
PAUSA_ENTRE_BUSQUEDAS = 3
PAUSA_ENTRE_DESCARGAS = 2
MAX_REINTENTOS = 2
MAX_ARCHIVOS_DESCARGAR = 40

# --------------------------------------------------------------------
# 📅 3. Rango de fechas de interés
FECHA_INICIO = "2023-01-01"
FECHA_FIN    = "2026-12-31"

# --------------------------------------------------------------------
# 📁 4. Carpetas de trabajo
CARPETA_BASE = "/content/icbf_contratos"
CARPETA_DESCARGAS = f"{CARPETA_BASE}/documentos_descargados"
os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

# --------------------------------------------------------------------
# 📊 5. Ruta del Excel — se crea AHORA, vacío, y se va llenando en vivo
RUTA_EXCEL = f"{CARPETA_BASE}/contratos_alimentos_ICBF_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

# --------------------------------------------------------------------
contador_busquedas_serpapi = 0

print("✅ Configuración cargada")
print(f"   - Máx. búsquedas SerpApi permitidas: {MAX_BUSQUEDAS_SERPAPI}")
print(f"   - Rango de fechas: {FECHA_INICIO} a {FECHA_FIN}")
print(f"   - Excel en vivo: {RUTA_EXCEL}")
print(f"   - Carpeta de descargas: {CARPETA_DESCARGAS}")

✅ Configuración cargada
   - Máx. búsquedas SerpApi permitidas: 10
   - Rango de fechas: 2023-01-01 a 2026-12-31
   - Excel en vivo: /content/icbf_contratos/contratos_alimentos_ICBF_20260618_0158.xlsx
   - Carpeta de descargas: /content/icbf_contratos/documentos_descargados


In [16]:
# CELDA 3 (CORREGIDA): Inicializar el Excel y crear la función de escritura EN VIVO
# Se agregó saneo de valores: algunos campos de SECOP llegan como diccionarios
# (ej: {"url": "https://...", "description": "..."}) y Excel NO acepta eso directamente.

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

ENCABEZADOS = ["Entidad", "Número de contrato", "Fecha", "Proveedor", "Valor",
               "Objeto del contrato", "Enlace de descarga", "Archivo local", "Fuente"]

def sanear_valor_excel(valor):
    """
    Convierte cualquier valor (incluyendo dict/list que a veces devuelve Socrata)
    en algo que Excel pueda recibir: texto, número o vacío.
    """
    if valor is None:
        return ""
    if isinstance(valor, dict):
        # Caso típico: {"url": "https://...", "description": "..."}
        if "url" in valor:
            return str(valor["url"])
        return str(valor)
    if isinstance(valor, list):
        return "; ".join(str(v) for v in valor)
    if isinstance(valor, (str, int, float, bool)):
        return valor
    return str(valor)

def inicializar_excel():
    wb = Workbook()
    ws = wb.active
    ws.title = "Contratos ICBF Alimentos"
    ws.append(ENCABEZADOS)
    for col_num, _ in enumerate(ENCABEZADOS, 1):
        celda = ws.cell(row=1, column=col_num)
        celda.font = Font(bold=True, color="FFFFFF", name="Arial")
        celda.fill = PatternFill("solid", start_color="1F4E78")
        celda.alignment = Alignment(horizontal="center")
    anchos = [25, 20, 15, 30, 15, 50, 45, 30, 30]
    for i, ancho in enumerate(anchos, 1):
        ws.column_dimensions[get_column_letter(i)].width = ancho
    wb.save(RUTA_EXCEL)

def agregar_fila_excel(entidad, numero_contrato, fecha, proveedor, valor, objeto, enlace, archivo_local, fuente):
    """
    Abre el Excel, agrega UNA fila (saneando cada valor), y lo guarda de inmediato.
    """
    wb = load_workbook(RUTA_EXCEL)
    ws = wb.active
    fila = [
        sanear_valor_excel(entidad),
        sanear_valor_excel(numero_contrato),
        sanear_valor_excel(fecha),
        sanear_valor_excel(proveedor),
        sanear_valor_excel(valor),
        sanear_valor_excel(objeto),
        sanear_valor_excel(enlace),
        sanear_valor_excel(archivo_local),
        sanear_valor_excel(fuente),
    ]
    ws.append(fila)
    for celda in ws[ws.max_row]:
        celda.font = Font(name="Arial", size=10)
    wb.save(RUTA_EXCEL)

inicializar_excel()
print(f"✅ Excel inicializado y listo para recibir datos en vivo: {RUTA_EXCEL}")
print("   👉 Puedes abrirlo/descargarlo en cualquier momento para ver el avance.")

✅ Excel inicializado y listo para recibir datos en vivo: /content/icbf_contratos/contratos_alimentos_ICBF_20260618_0158.xlsx
   👉 Puedes abrirlo/descargarlo en cualquier momento para ver el avance.


In [17]:
# CELDA 4: Funciones de utilidad

import requests
from tqdm.notebook import tqdm

def solicitud_segura(url, params=None, headers=None, timeout=TIMEOUT_REQUEST, reintentos=MAX_REINTENTOS, silencioso=False):
    intento = 0
    while intento <= reintentos:
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp
            elif not silencioso:
                print(f"   ⚠️ Código {resp.status_code} (intento {intento+1})")
        except requests.exceptions.Timeout:
            if not silencioso:
                print(f"   ⏱️ Timeout (intento {intento+1})")
        except requests.exceptions.RequestException as e:
            if not silencioso:
                print(f"   ⚠️ Error de red (intento {intento+1}): {e}")

        intento += 1
        if intento <= reintentos:
            time.sleep(3 * intento)

    return None

print("✅ Funciones de utilidad listas")

✅ Funciones de utilidad listas


In [18]:
# CELDA 5 (CORREGIDA): Búsqueda en SECOP — ahora detecta el esquema real de cada
# dataset ANTES de construir el filtro, para evitar el error 400 "Bad Request"
# (ese error ocurría porque SECOP I usa nombres de columna distintos a SECOP II)

DATASETS_SECOP = {
    "SECOP II - Contratos Electrónicos": "jbjy-vk9h",
    "SECOP I - Contratos": "scjd-4sr4",
    "SECOP I - Contratos +16K SMMLV": "79ga-5jck",
}

PALABRAS_CLAVE_ALIMENTOS = ["alimento", "alimentaria", "alimentacion", "nutricion", "raciones", "minuta"]

# Posibles nombres de columna según el dataset (orden de preferencia)
POSIBLES_COLUMNAS = {
    "entidad": ["nombre_entidad", "entidad", "nombre_de_la_entidad"],
    "fecha": ["fecha_de_firma", "fecha_de_firma_del_contrato", "fecha_de_cargue_en_el_secop",
              "fecha_de_inicio_del_contrato", "fechaduracioncontrato"],
    "objeto": ["objeto_del_contrato", "detalle_del_objeto_a_contratar", "objeto", "objeto_a_contratar"],
    "numero_contrato": ["id_contrato", "numero_del_contrato", "uid", "numero_de_proceso"],
    "proveedor": ["nom_raz_social_contratista", "nombre_del_proveedor", "proveedor_adjudicado", "nombre_razon_social_contratista"],
    "valor": ["valor_del_contrato", "cuantia_contrato", "valor_contrato"],
    "enlace": ["urlproceso", "url_contrato", "documento_proceso", "url_del_proceso"],
}

def obtener_columnas_reales(dataset_id):
    """Consulta el esquema real del dataset para saber qué columnas existen de verdad."""
    url_metadata = f"https://www.datos.gov.co/api/views/{dataset_id}.json"
    resp = solicitud_segura(url_metadata, timeout=15, silencioso=True)
    if resp is None:
        return []
    try:
        metadata = resp.json()
        return [col["fieldName"] for col in metadata.get("columns", [])]
    except Exception:
        return []

def mapear_columnas(columnas_reales):
    """Para cada campo que necesitamos, elige el primer nombre que SÍ exista en este dataset."""
    mapeo = {}
    for campo, posibles in POSIBLES_COLUMNAS.items():
        encontrado = next((p for p in posibles if p in columnas_reales), None)
        mapeo[campo] = encontrado
    return mapeo

def primera_clave_disponible(registro, posibles):
    for c in posibles:
        if c and c in registro and registro[c]:
            return registro[c]
    return ""

print("="*70)
print("BUSCANDO CONTRATOS EN SECOP (I y II)")
print("="*70)

total_secop_encontrados = 0

for nombre_ds, ds_id in tqdm(DATASETS_SECOP.items(), desc="Datasets SECOP", total=len(DATASETS_SECOP)):
    print(f"\n🔎 Consultando: {nombre_ds}")

    # 1) Primero detectamos qué columnas existen REALMENTE en este dataset
    columnas_reales = obtener_columnas_reales(ds_id)
    if not columnas_reales:
        print(f"   ⚠️ No se pudo obtener el esquema de {nombre_ds}. Se omite.")
        time.sleep(2)
        continue

    mapeo = mapear_columnas(columnas_reales)
    print(f"   🗂️ Columnas detectadas: entidad={mapeo['entidad']}, fecha={mapeo['fecha']}, objeto={mapeo['objeto']}")

    if not mapeo["entidad"] or not mapeo["objeto"]:
        print(f"   ⚠️ Este dataset no tiene columnas reconocibles de entidad/objeto. Se omite.")
        time.sleep(2)
        continue

    # 2) Construimos el filtro SOLO con columnas que existen de verdad
    condiciones_alimentos = [f"upper({mapeo['objeto']}) like '%{p.upper()}%'" for p in PALABRAS_CLAVE_ALIMENTOS]
    filtro_alimentos = "(" + " OR ".join(condiciones_alimentos) + ")"

    partes_where = [f"upper({mapeo['entidad']}) like '%ICBF%'", filtro_alimentos]

    # La fecha solo se filtra en el servidor SI existe esa columna; si no, filtramos luego en Python
    filtrar_fecha_en_servidor = mapeo["fecha"] is not None
    if filtrar_fecha_en_servidor:
        partes_where.append(f"{mapeo['fecha']} >= '{FECHA_INICIO}T00:00:00'")
        partes_where.append(f"{mapeo['fecha']} <= '{FECHA_FIN}T23:59:59'")

    where_clause = " AND ".join(partes_where)
    url = f"https://www.datos.gov.co/resource/{ds_id}.json"
    params = {"$where": where_clause, "$limit": 50}
    headers = {"User-Agent": "Curso-Contraloria-WebScraping/1.0"}

    resp = solicitud_segura(url, params=params, headers=headers)

    # 3) Si aun así falla (p.ej. el tipo de columna de fecha es texto, no datetime),
    #    reintentamos SIN el filtro de fecha y filtramos después en Python
    if resp is None and filtrar_fecha_en_servidor:
        print("   🔁 Reintentando sin filtro de fecha en el servidor...")
        where_clause_simple = " AND ".join(partes_where[:2])
        params = {"$where": where_clause_simple, "$limit": 100}
        resp = solicitud_segura(url, params=params, headers=headers)
        filtrar_fecha_en_servidor = False  # ahora toca filtrar en Python

    if resp is None:
        print(f"   ❌ No fue posible consultar {nombre_ds}. Se continúa con el siguiente.")
        time.sleep(2)
        continue

    try:
        registros = resp.json()
    except Exception as e:
        print(f"   ⚠️ Error procesando respuesta: {e}")
        continue

    if not registros:
        print("   ℹ️ Sin resultados con estos filtros.")
        time.sleep(2)
        continue

    # 4) Si no se filtró fecha en el servidor, lo hacemos aquí (best-effort)
    if not filtrar_fecha_en_servidor and mapeo["fecha"]:
        def fecha_en_rango(reg):
            f = reg.get(mapeo["fecha"], "")
            return FECHA_INICIO <= str(f)[:10] <= FECHA_FIN if f else True
        registros = [r for r in registros if fecha_en_rango(r)]

    print(f"   ✅ {len(registros)} contratos encontrados. Escribiendo en Excel en vivo...")

    for reg in tqdm(registros, desc=f"   Guardando {nombre_ds}", leave=False):
        entidad = primera_clave_disponible(reg, [mapeo["entidad"]])
        numero_contrato = primera_clave_disponible(reg, [mapeo["numero_contrato"]])
        fecha = primera_clave_disponible(reg, [mapeo["fecha"]])
        proveedor = primera_clave_disponible(reg, [mapeo["proveedor"]])
        valor = primera_clave_disponible(reg, [mapeo["valor"]])
        objeto = primera_clave_disponible(reg, [mapeo["objeto"]])
        enlace = primera_clave_disponible(reg, [mapeo["enlace"]])

        agregar_fila_excel(entidad, numero_contrato, fecha, proveedor, valor, objeto, enlace, "", nombre_ds)
        total_secop_encontrados += 1

    time.sleep(2)

print(f"\n📊 TOTAL contratos de SECOP ya escritos en el Excel: {total_secop_encontrados}")

BUSCANDO CONTRATOS EN SECOP (I y II)


Datasets SECOP:   0%|          | 0/3 [00:00<?, ?it/s]


🔎 Consultando: SECOP II - Contratos Electrónicos
   🗂️ Columnas detectadas: entidad=nombre_entidad, fecha=fecha_de_firma, objeto=objeto_del_contrato
   ⏱️ Timeout (intento 1)
   ✅ 50 contratos encontrados. Escribiendo en Excel en vivo...


   Guardando SECOP II - Contratos Electrónicos:   0%|          | 0/50 [00:00<?, ?it/s]


🔎 Consultando: SECOP I - Contratos
   🗂️ Columnas detectadas: entidad=nombre_de_la_entidad, fecha=fecha_de_firma_del_contrato, objeto=detalle_del_objeto_a_contratar
   ℹ️ Sin resultados con estos filtros.

🔎 Consultando: SECOP I - Contratos +16K SMMLV
   🗂️ Columnas detectadas: entidad=nombre_de_la_entidad, fecha=fecha_de_firma_del_contrato, objeto=detalle_del_objeto_a_contratar
   ℹ️ Sin resultados con estos filtros.

📊 TOTAL contratos de SECOP ya escritos en el Excel: 50


In [19]:
# CELDA 6: Búsqueda complementaria en Google vía SerpApi
# Tope DURO de 10 búsquedas. Cada resultado se escribe en el Excel al instante.

from serpapi import GoogleSearch

terminos_busqueda = [
    "ICBF contrato alimentos adjudicación 2026",
    "ICBF contrato alimentos adjudicación 2025",
    "ICBF contrato alimentos adjudicación 2024",
    "ICBF contrato alimentos adjudicación 2023",
    "ICBF resolución adjudicación raciones alimentarias",
    "ICBF contrato suministro alimentos filetype:pdf",
    "ICBF minuta operador alimentación escolar contrato",
    "ICBF proceso licitación alimentos 2025",
    "ICBF contrato alimentación primera infancia",
    "ICBF adjudicación complementación alimentaria",
]

print("="*70)
print(f"BUSCANDO EN GOOGLE VÍA SERPAPI (máximo {MAX_BUSQUEDAS_SERPAPI} búsquedas)")
print("="*70)

total_serpapi_encontrados = 0
barra_serpapi = tqdm(terminos_busqueda, desc="Búsquedas SerpApi")

for termino in barra_serpapi:
    if contador_busquedas_serpapi >= MAX_BUSQUEDAS_SERPAPI:
        print(f"\n🛑 Límite de {MAX_BUSQUEDAS_SERPAPI} búsquedas alcanzado. Deteniendo SerpApi.")
        break

    barra_serpapi.set_postfix_str(f"{contador_busquedas_serpapi+1}/{MAX_BUSQUEDAS_SERPAPI}: {termino[:30]}...")

    params = {
        "engine": "google", "q": termino, "api_key": SERPAPI_KEY,
        "num": 10, "hl": "es", "gl": "co",
    }

    try:
        search = GoogleSearch(params)
        data = search.get_dict()
    except Exception as e:
        print(f"   ⚠️ Error en SerpApi: {e}")
        contador_busquedas_serpapi += 1
        continue

    contador_busquedas_serpapi += 1
    organicos = data.get("organic_results", [])

    if not organicos:
        print(f"   ℹ️ Sin resultados para: {termino}")
    else:
        for r in organicos:
            agregar_fila_excel(
                entidad="ICBF",
                numero_contrato="No identificado (verificar manualmente)",
                fecha=datetime.now().strftime("%Y-%m-%d"),
                proveedor="",
                valor="",
                objeto=r.get("title", ""),
                enlace=r.get("link", ""),
                archivo_local="",
                fuente=f"SerpApi: {termino}"
            )
            total_serpapi_encontrados += 1
        print(f"   ✅ {len(organicos)} resultados escritos en el Excel")

    time.sleep(PAUSA_ENTRE_BUSQUEDAS)

print(f"\n📊 Búsquedas SerpApi usadas: {contador_busquedas_serpapi}/{MAX_BUSQUEDAS_SERPAPI}")
print(f"📊 TOTAL resultados web escritos en el Excel: {total_serpapi_encontrados}")

BUSCANDO EN GOOGLE VÍA SERPAPI (máximo 10 búsquedas)


Búsquedas SerpApi:   0%|          | 0/10 [00:00<?, ?it/s]

   ✅ 9 resultados escritos en el Excel
   ✅ 9 resultados escritos en el Excel
   ✅ 9 resultados escritos en el Excel
   ✅ 10 resultados escritos en el Excel
   ✅ 9 resultados escritos en el Excel
   ✅ 10 resultados escritos en el Excel
   ✅ 10 resultados escritos en el Excel
   ✅ 10 resultados escritos en el Excel
   ✅ 10 resultados escritos en el Excel
   ✅ 10 resultados escritos en el Excel

📊 Búsquedas SerpApi usadas: 10/10
📊 TOTAL resultados web escritos en el Excel: 96


In [20]:
# CELDA 7: Descarga de documentos — cada descarga se refleja al instante en el Excel
# (se actualiza la columna "Archivo local" de la fila correspondiente)

import re
from urllib.parse import urlparse

def nombre_archivo_seguro(texto, indice):
    base = re.sub(r'[^a-zA-Z0-9_-]', '_', str(texto))[:40]
    return f"{indice:03d}_{base}"

def es_descargable(url):
    extensiones = (".pdf", ".doc", ".docx", ".xls", ".xlsx", ".zip")
    return isinstance(url, str) and url.lower().endswith(extensiones)

def actualizar_archivo_local_en_excel(fila_excel, nombre_archivo):
    """Actualiza la columna 'Archivo local' de una fila específica ya escrita."""
    wb = load_workbook(RUTA_EXCEL)
    ws = wb.active
    ws.cell(row=fila_excel, column=8, value=nombre_archivo)  # columna 8 = "Archivo local"
    wb.save(RUTA_EXCEL)

print("="*70)
print("DESCARGANDO DOCUMENTOS ENCONTRADOS")
print("="*70)

# Releemos el Excel para saber qué enlaces son descargables y en qué fila están
wb_lectura = load_workbook(RUTA_EXCEL)
ws_lectura = wb_lectura.active

candidatos = []
for fila_idx in range(2, ws_lectura.max_row + 1):
    url = ws_lectura.cell(row=fila_idx, column=7).value   # columna 7 = "Enlace de descarga"
    objeto = ws_lectura.cell(row=fila_idx, column=6).value
    if es_descargable(url):
        candidatos.append((fila_idx, url, objeto))

print(f"📄 {len(candidatos)} enlaces parecen ser archivos descargables (de {ws_lectura.max_row - 1} filas totales)")

contador_descargas = 0
barra_descargas = tqdm(candidatos, desc="Descargando documentos")

for fila_idx, url, objeto in barra_descargas:
    if contador_descargas >= MAX_ARCHIVOS_DESCARGAR:
        print(f"\n🛑 Límite de {MAX_ARCHIVOS_DESCARGAR} descargas alcanzado.")
        break

    barra_descargas.set_postfix_str(f"{contador_descargas+1}/{MAX_ARCHIVOS_DESCARGAR}")
    resp = solicitud_segura(url, silencioso=True)

    if resp is None:
        continue

    extension = os.path.splitext(urlparse(url).path)[1] or ".bin"
    nombre = nombre_archivo_seguro(objeto or "documento", fila_idx) + extension
    ruta_local = os.path.join(CARPETA_DESCARGAS, nombre)

    try:
        with open(ruta_local, "wb") as f:
            f.write(resp.content)
        actualizar_archivo_local_en_excel(fila_idx, nombre)  # 👈 se refleja YA en el Excel
        contador_descargas += 1
    except Exception as e:
        print(f"   ⚠️ No se pudo guardar: {e}")

    time.sleep(PAUSA_ENTRE_DESCARGAS)

print(f"\n📊 TOTAL archivos descargados y reflejados en el Excel: {contador_descargas}")

DESCARGANDO DOCUMENTOS ENCONTRADOS
📄 31 enlaces parecen ser archivos descargables (de 146 filas totales)


Descargando documentos:   0%|          | 0/31 [00:00<?, ?it/s]


📊 TOTAL archivos descargados y reflejados en el Excel: 31


In [21]:
# CELDA 8: Resumen final + descarga del Excel y del .zip con los documentos

from google.colab import files
import shutil

wb_final = load_workbook(RUTA_EXCEL)
ws_final = wb_final.active
total_filas = ws_final.max_row - 1

print("="*70)
print("RESUMEN DEL EJERCICIO")
print("="*70)
print(f"📊 Total de registros en el Excel: {total_filas}")
print(f"📥 Total de archivos descargados: {contador_descargas}")
print(f"🔑 Búsquedas SerpApi usadas: {contador_busquedas_serpapi}/{MAX_BUSQUEDAS_SERPAPI}")
print(f"📁 Excel: {RUTA_EXCEL}")
print(f"📁 Documentos: {CARPETA_DESCARGAS}")

# Comprimir documentos descargados
ruta_zip_base = f"{CARPETA_BASE}/documentos_ICBF"
shutil.make_archive(ruta_zip_base, 'zip', CARPETA_DESCARGAS)

files.download(RUTA_EXCEL)
files.download(f"{ruta_zip_base}.zip")

print("\n✅ Descarga iniciada.")

RESUMEN DEL EJERCICIO
📊 Total de registros en el Excel: 146
📥 Total de archivos descargados: 31
🔑 Búsquedas SerpApi usadas: 10/10
📁 Excel: /content/icbf_contratos/contratos_alimentos_ICBF_20260618_0158.xlsx
📁 Documentos: /content/icbf_contratos/documentos_descargados


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Descarga iniciada.
